# OpenManus on Kaggle - free LLM APIs, automatic model discoveryThis notebook runs the **OpenManus** agent (`himanshuverma993/OpenManus`) on a Kaggle CPU notebook using **free** LLM providers.**No model is ever selected by hand.** OpenManus asks each provider for the models that are available *right now*, keeps the ones that are free and can do tool calling, and rotates to the next provider whenever the current one is unavailable, rate limited, exhausted or incompatible.```Groq  ─┐Gemini ─┼─►  discover  ─►  rank (free + tool capable)  ─►  use best  ─►  fail?  ─►  next providerOpenRouter ─┘```Everything below is self-contained: it installs the slim dependency set, clones the repo, resolves your API keys from **Kaggle Secrets**, writes `config/config.toml` and runs the agent.## What you need before running1. A free API key from **one** of (more is better - they are used as a fallback chain):   | Provider | Key | Free tier notes |   |---|---|---|   | [Groq](https://console.groq.com/keys) | `GROQ_API_KEY` | fastest free tier, per-model rate limits |   | [Google Gemini](https://aistudio.google.com/apikey) | `GEMINI_API_KEY` | free RPM/RPD quota, 1M context, vision |   | [OpenRouter](https://openrouter.ai/keys) | `OPENROUTER_API_KEY` | `:free` models, but only ~20 req/min and ~50 req/day |2. In Kaggle: **Add-ons -> Secrets -> New Secret** and add the key with the exact name (`GROQ_API_KEY`, `GEMINI_API_KEY`, `OPENROUTER_API_KEY`).3. Enable **Internet** in the notebook settings (right-hand side panel) - Kaggle needs it for `pip` and for the API calls.

## 1 - Clone the repo and install the dependencies

In [ ]:
import osimport subprocessimport sysREPO_DIR = "/kaggle/working/OpenManus"REPO_URL = "https://github.com/himanshuverma993/OpenManus.git"if not os.path.isdir(os.path.join(REPO_DIR, ".git")):    subprocess.run(        ["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],        check=True,    )else:    print("Repo already present ->", REPO_DIR)os.chdir(REPO_DIR)if REPO_DIR not in sys.path:    sys.path.insert(0, REPO_DIR)print("cwd:", os.getcwd())

In [ ]:
# The slim dependency set: only what the core Manus agent needs.# (skips playwright / browsergym / crawl4ai / docker / boto3 / pandas)!pip install -q --upgrade pip!pip install -q -r requirements-kaggle.txt# Pre-warm the tiktoken BPE cache so token counting is accurate.# If the download is blocked the agent still works, just with rough token counts.import tiktokentry:    tiktoken.get_encoding("cl100k_base")    print("tiktoken cache ready")except Exception as exc:    print("tiktoken cache unavailable (offline tokenizer will be used):", exc)

## 2 - Resolve the API keysEvery key is optional on its own: OpenManus uses the providers whose key it finds and skips the others. The order of `PROVIDER_ORDER` is the fallback order.

In [ ]:
import osPROVIDER_ORDER = ["groq", "gemini", "openrouter"]SECRET_NAMES = {    "groq": "GROQ_API_KEY",    "gemini": "GEMINI_API_KEY",    "openrouter": "OPENROUTER_API_KEY",}def resolve_api_key(secret_name: str) -> str:    """Kaggle Secret -> environment variable -> empty string."""    try:        from kaggle_secrets import UserSecretsClient        value = UserSecretsClient().get_secret(secret_name)        if value:            os.environ[secret_name] = value            return value    except Exception as exc:        print(f"[{secret_name}] secret not available ({type(exc).__name__})")    return os.environ.get(secret_name, "")api_keys = {name: resolve_api_key(SECRET_NAMES[name]) for name in PROVIDER_ORDER}for name in PROVIDER_ORDER:    status = "found" if api_keys[name] else "MISSING (provider will be skipped)"    print(f"{name:12} {SECRET_NAMES[name]:20} {status}")if not any(api_keys.values()):    raise SystemExit(        "No API key found. Add at least one of "        + ", ".join(SECRET_NAMES.values())        + " as a Kaggle secret and re-run this cell."    )

## 3 - Write `config/config.toml``config/config.toml` is git-ignored, so it does not exist after cloning - this cell creates it.The important part is `model = "auto"`: OpenManus discovers the usable models at runtime instead of using a hard coded name. The keys are referenced through environment placeholders, so no secret is ever written to disk.

In [ ]:
config_toml = """[llm]model = "auto"providers = ["groq", "gemini", "openrouter"]max_tokens = 8192temperature = 0.0max_retries = 2[providers.groq]api_key = "${GROQ_API_KEY}"max_candidates = 3free_only = true[providers.gemini]api_key = "${GEMINI_API_KEY}"max_candidates = 2free_only = true[providers.openrouter]api_key = "${OPENROUTER_API_KEY}"max_candidates = 2free_only = true[search]engine = "DuckDuckGo"fallback_engines = ["Baidu", "Bing", "Google"]retry_delay = 30max_retries = 2[sandbox]use_sandbox = false[mcp]server_reference = "app.mcp.server""""# Browser Use needs a local Chromium and `uvx`; neither exists on Kaggle, so# the MCP browser server is disabled explicitly (the agent works fine without).os.environ["OPENMANUS_DISABLE_BROWSER_USE"] = "1"os.makedirs("config", exist_ok=True)with open(os.path.join("config", "config.toml"), "w") as handle:    handle.write(config_toml)print(open(os.path.join("config", "config.toml")).read())

## 4 - Check what is actually available right nowThis is the runtime discovery step. It queries the `/models` endpoint of every provider with a key, filters the catalogue down to the free models that can do tool calling and prints the resulting fallback chain. Nothing is guessed - the list comes from the providers themselves.

In [ ]:
from app.providers import ProviderSpec, select_candidatesfrom app.config import configchain = config.llm["default"].providersprint("Configured provider chain:", [p.name for p in chain])print()usable = []for provider in chain:    if not provider.api_key:        print(f"[{provider.name}] skipped - no API key")        continue    spec = ProviderSpec(**provider.model_dump())    try:        candidates = select_candidates(spec, use_cache=False)    except Exception as exc:        print(f"[{provider.name}] unavailable - {exc}")        continue    usable.extend(candidates)    print(f"[{provider.name}] {len(candidates)} usable model(s):")    for candidate in candidates:        print(            f"    - {candidate.model}"            f"  (max output tokens: {candidate.max_completion_tokens or 'unknown'},"            f" images: {candidate.supports_images})"        )if not usable:    raise SystemExit("No provider could be reached - check the API keys and the internet setting.")print()print("Fallback chain that OpenManus will use:")for index, candidate in enumerate(usable, start=1):    print(f"  {index}. {candidate.label}")

## 5 - Run the agent`run_agent()` is a thin wrapper around the stock `Manus` agent, so nothing about the agent's behaviour is changed - it only adds notebook-friendly printing and cleanup.

In [ ]:
import asynciofrom app.agent.manus import Manusasync def run_agent(prompt: str) -> str:    """Run the OpenManus agent and return the final result."""    agent = await Manus.create()    try:        return await agent.run(prompt)    finally:        await agent.cleanup()def ask(prompt: str) -> str:    """Synchronous helper so the agent can be called from any cell."""    return asyncio.run(run_agent(prompt))print("Ready. Call ask(...) in the next cell.")

In [ ]:
# Change the text to whatever you want the agent to do.result = ask(    "Compute the first 25 Fibonacci numbers with python, then tell me the sum.")print("\n" + "=" * 70)print("FINAL RESULT")print("=" * 70)print(result)

## Troubleshooting| Symptom | Cause / fix ||---|---|| `ValidationError: daytona_api_key` | Fixed in this branch - `DaytonaSettings` defaults to an empty key, so a `config.toml` without a `[daytona]` section loads fine. || `StdinNotImplementedError` / notebook hangs | Fixed - `ask_human` detects a non-interactive stdin and returns a message instead of calling `input()`. || `ModuleNotFoundError: boto3` / `docker` | Fixed - both are imported lazily now; neither is needed for a text agent. || `No usable LLM provider could be reached` | No key was found, or every provider rejected the key. Re-check the secret names and the *Internet* toggle in the notebook settings. || `401 Unauthorized` | Wrong or missing key. The chain rotates to the next provider, but if that is the only key, nothing can work. || `429 Too Many Requests` | Free-tier limit. OpenManus rotates to the next provider automatically; lower `max_steps` if all providers are throttled. || `400 Bad Request` on `max_tokens` | Fixed automatically - the output budget is clamped to the model's own limit, which is read from the provider. || The agent stops after ~20 steps | The default `max_steps` of the stock agent. Raise it in `Manus.create(max_steps=...)` if the task needs more. |## Notes on the free tiers- **Groq** - the free tier allows all currently published models, but the rate limits are per model and per account. The catalogue changes often, which is exactly why the model is discovered at runtime instead of being hard coded.- **Gemini** - the free tier has per-model RPM/RPD quotas. Gemini is also the best choice for the vision tools (`supports_images`).- **OpenRouter** - without purchased credits the free models are limited to roughly 20 requests/minute and 50 requests/day, so a multi-step agent run exhausts them quickly. Keep Groq or Gemini in the chain.